# Pick & Go — Entraînement YOLOv8 (Google Colab)

**Instructions :**
1. `Exécution > Modifier le type d'exécution` → choisir **GPU T4**
2. Exécuter toutes les cellules dans l'ordre
3. Télécharger `best.pt` à la fin et le placer dans `models/pick_and_go/weights/`

In [ ]:
# 1. Installation des dépendances
!pip install ultralytics icrawler -q

In [ ]:
# 2. Catalogue des produits et configuration
PRODUCTS = {
    'Bouteille_Eau': {'nom': "Bouteille d'Eau",  'prix': 300},
    'Cahier':        {'nom': 'Cahier',            'prix': 500},
    'Stylo':         {'nom': 'Stylo',             'prix': 200},
    'Gazelle':       {'nom': 'Biere Gazelle',     'prix': 800},
    'Choco_Pain':    {'nom': 'Choco Pain',        'prix': 250},
    'Bissap':        {'nom': 'Jus Bissap',        'prix': 400},
    'Riz':           {'nom': 'Sac de Riz',        'prix': 1500},
    'Huile':         {'nom': 'Huile',             'prix': 2000},
}

SEARCH_QUERIES = {
    'Bouteille_Eau': 'bouteille eau minerale plastique transparent',
    'Cahier':        'cahier scolaire spirale lignes',
    'Stylo':         'stylo bille bleu noir ecriture',
    'Gazelle':       'gazelle biere bouteille senegal',
    'Choco_Pain':    'pain au chocolat viennoiserie boulangerie',
    'Bissap':        'bissap jus hibiscus sachet boisson senegal',
    'Riz':           'sac riz 5kg grains senegal',
    'Huile':         'huile cuisine bouteille jaune litre',
}

IMAGES_PER_CLASS = 80
TRAIN_RATIO = 0.8
print('Configuration chargee.')

In [ ]:
# 3. Collecte des images
import os, shutil, random
from pathlib import Path
from icrawler.builtin import BingImageCrawler

RAW = Path('data/raw')

for class_name in PRODUCTS:
    save_dir = RAW / class_name
    save_dir.mkdir(parents=True, exist_ok=True)
    existing = list(save_dir.glob('*.*'))
    if len(existing) >= IMAGES_PER_CLASS:
        print(f'[OK] {class_name}: deja {len(existing)} images.')
        continue
    print(f'[DL] {class_name}...')
    crawler = BingImageCrawler(
        feeder_threads=2, parser_threads=2, downloader_threads=4,
        storage={'root_dir': str(save_dir)}
    )
    crawler.crawl(keyword=SEARCH_QUERIES[class_name], max_num=IMAGES_PER_CLASS, min_size=(100,100))
    print(f'  -> {len(list(save_dir.glob("*.*")))} images')

print('Collecte terminee.')

In [ ]:
# 4. Annotation automatique YOLO + split train/val
VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def write_label(path, class_id):
    Path(path).write_text(f'{class_id} 0.5 0.5 0.9 0.9\n')

for dest in ['data/train/images','data/train/labels','data/val/images','data/val/labels']:
    Path(dest).mkdir(parents=True, exist_ok=True)

class_names = list(PRODUCTS.keys())
total_train = total_val = 0

for class_id, class_name in enumerate(class_names):
    images = [f for f in (RAW/class_name).iterdir() if f.suffix.lower() in VALID_EXT]
    random.shuffle(images)
    split = int(len(images) * TRAIN_RATIO)
    for img in images[:split]:
        stem = f'{class_name}_{img.stem}'
        shutil.copy2(img, f'data/train/images/{stem}{img.suffix}')
        write_label(f'data/train/labels/{stem}.txt', class_id)
    for img in images[split:]:
        stem = f'{class_name}_{img.stem}'
        shutil.copy2(img, f'data/val/images/{stem}{img.suffix}')
        write_label(f'data/val/labels/{stem}.txt', class_id)
    print(f'  {class_name}: {split} train | {len(images)-split} val')
    total_train += split; total_val += len(images)-split

print(f'Dataset pret: {total_train} train | {total_val} val')

In [ ]:
# 5. Création du data.yaml
yaml_content = f"""path: /content
train: data/train/images
val:   data/val/images

nc: {len(PRODUCTS)}
names:\n"""
for name in PRODUCTS:
    yaml_content += f'  - {name}\n'

with open('data.yaml', 'w') as f:
    f.write(yaml_content)
print('data.yaml cree:')
print(yaml_content)

In [ ]:
# 6. Entraînement YOLOv8 sur GPU
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='models',
    name='pick_and_go',
    exist_ok=True,
)
print('Entrainement termine!')
print('Meilleur modele: models/pick_and_go/weights/best.pt')

In [ ]:
# 7. Téléchargement du modèle entraîné
from google.colab import files
files.download('models/pick_and_go/weights/best.pt')
print('Placez best.pt dans : models/pick_and_go/weights/best.pt')